# Convex-combos 02378 train-mean-centered regular-L1 analysis

This notebook analyzes the digits 0,2,3,7,8 MNIST convex-combo regular-L1 run trained with a displacement center computed from the **training split only**. The active checkpoint is seed 0 from the deterministic `train_mean` sweep: a leakage-free `c=1e-4` cosine-scheduler run selected for strong nearest-neighbor recovery (24/25 restricted-digit neighbors).

Important split note: this experiment does **not** hold out whole digit classes. Training uses a random sample split of the synthetic convex-combo maps (`test_fraction=0.1`, `seed=42`), so all five digit atoms appear in both train and test mixtures.

The active trial uses the `displacement_centered` model with `displacement_center_mode=train_mean` and `model_seed=0`: it encodes residuals `T - X - mu_train` and adds the fixed training mean displacement back during decoding. It includes atom matching, coefficient recovery heatmaps, reconstruction examples, source-to-reconstruction overlays, matched atom vector fields, and nearest-held-out MNIST transport-map comparisons.


In [ ]:

%matplotlib inline

from pathlib import Path
import json
import math
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.optimize import linear_sum_assignment

# Find the repository root from the current kernel working directory.
cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (candidate for candidate in [cwd, *cwd.parents]
     if (candidate / "mnist").exists() and (candidate / "datasets").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Run this notebook from the repository or one of its subdirectories.")

MNIST_PIPELINE_DIR = REPO_ROOT / "mnist" / "pipeline"
for path in (REPO_ROOT, MNIST_PIPELINE_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from mnist_sae_models import DisplacementFieldSAE, TransportMapSAE, load_model_state
from pointcloud.pipeline.pointcloud_centered_sae import CenteredDisplacementFieldSAE

TRIAL_NAME = "convex_combos_02378"
RESULT_SET = "convex_combos_02378_train_mean_seed_sweep"
FIG_PREFIX = TRIAL_NAME
RESULT_DIR = REPO_ROOT / "mnist" / "results" / RESULT_SET / "final" / "seed_000"
CKPT_NAME = "displacement_centered_eps0.025_c0.0001_best.pt"
CKPT_PATH = RESULT_DIR / CKPT_NAME
CONFIG_PATH = RESULT_DIR / "config.json"
DATA_DIR = REPO_ROOT / "datasets" / "convex_combos_02378"
MNIST_OT_DIR = REPO_ROOT / "datasets" / "mnist_ot"
OUT_DIR = RESULT_DIR / "notebook_figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
BATCH_SIZE = 1024

mpl.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
})

ALL_DIGIT_COLORS = {d: mpl.colormaps["tab10"](d) for d in range(10)}
RECON_COLOR = "#D95F02"
TARGET_COLOR = "#1F1F1F"
BASE_COLOR = "#B8B8B8"


def l2rho_sq_np(a, b):
    """Squared L2(rho): mean_x ||a(x) - b(x)||_2^2."""
    diff = np.asarray(a) - np.asarray(b)
    return np.mean(np.sum(diff * diff, axis=-1), axis=-1)


def simplex_normalize(codes, eps=1e-12):
    row_sum = codes.sum(axis=1, keepdims=True)
    return codes / np.clip(row_sum, eps, None)


def train_test_indices(n, test_fraction, seed):
    """Replicate torch.utils.data.random_split(dataset, [n_train, n_test])."""
    n_test = int(n * test_fraction)
    n_train = n - n_test
    gen = torch.Generator().manual_seed(int(seed))
    perm = torch.randperm(n, generator=gen).numpy()
    return perm[:n_train], perm[n_train:]


def load_convex_combo_dataset(data_dir):
    data_dir = Path(data_dir)
    X = torch.load(data_dir / "base_measure.pt", map_location="cpu").float()
    maps = torch.load(data_dir / "maps" / "maps.pt", map_location="cpu").float()
    coeffs = torch.load(data_dir / "coefficients" / "coefficients.pt", map_location="cpu").float()
    true_atoms = torch.load(data_dir / "base_maps.pt", map_location="cpu").float()
    meta = json.loads((data_dir / "metadata.json").read_text())
    digits = [int(d) for d in meta["digits"]]
    return X, maps, coeffs, true_atoms, meta, digits


def infer_atoms_type(state_dict):
    if "atoms_module.H_raw" in state_dict:
        return "gibbs"
    if "atoms_module.psi" in state_dict:
        return "sinkhorn"
    raise ValueError("Could not infer atom parameterization from checkpoint state_dict.")


def rebuild_model(ckpt_path, config_path, device=DEVICE):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    config = json.loads(Path(config_path).read_text())
    state = ckpt["model_state"]
    method = ckpt.get("method", "displacement")
    grid_points = ckpt.get("grid_points")
    model_kwargs = dict(
        m=int(ckpt["m"]),
        eps=float(ckpt["eps"]),
        grid_side=int(config.get("grid_side", 32)),
        lista_steps=int(ckpt["lista_steps"]),
        activation_type=config.get("activation_type", "relu"),
        normalize_atoms=True,
        grid_points=grid_points.float() if grid_points is not None else None,
        atoms_type=infer_atoms_type(state),
        n_sinkhorn=int(config.get("n_sinkhorn", 30)),
        topk_k=int(config.get("topk_k", 3)),
        per_atom_gain=("encoder.gain" in state),
        lateral_init="damped_identity",
    )
    if method == "raw_map":
        model = TransportMapSAE(ckpt["X"].float(), **model_kwargs).to(device)
    elif method == "displacement":
        model = DisplacementFieldSAE(ckpt["X"].float(), **model_kwargs).to(device)
    elif method == "displacement_centered":
        model = CenteredDisplacementFieldSAE(
            ckpt["X"].float(), ckpt["displacement_center"].float(), **model_kwargs,
        ).to(device)
    elif method == "displacement_centered_pc_removed":
        model = PCRemovedCenteredDisplacementFieldSAE(
            ckpt["X"].float(),
            ckpt["displacement_center"].float(),
            ckpt["displacement_pc_component"].float(),
            **model_kwargs,
        ).to(device)
    elif method == "displacement_centered_whitened":
        model = WhitenedCenteredDisplacementFieldSAE(
            ckpt["X"].float(),
            ckpt["displacement_center"].float(),
            ckpt["whitening_matrix"].float(),
            ckpt["unwhitening_matrix"].float(),
            **model_kwargs,
        ).to(device)
    else:
        raise ValueError(f"Unknown checkpoint method: {method}")
    load_model_state(model, state)
    model.eval()
    return model, ckpt, config


@torch.no_grad()
def forward_all(model, maps, batch_size=BATCH_SIZE, device=DEVICE):
    recons = []
    codes = []
    recon_losses = []
    n = model.n
    for start in range(0, maps.shape[0], batch_size):
        batch = maps[start:start + batch_size].to(device).float()
        recon, lam = model(batch)
        # Training convention: 0.5 * ||T - T_hat||^2_{L2(rho)}.
        loss = 0.5 * ((batch - recon) ** 2).sum(dim=(1, 2)) / n
        recons.append(recon.cpu())
        codes.append(lam.cpu())
        recon_losses.append(loss.cpu())
    return torch.cat(recons), torch.cat(codes), torch.cat(recon_losses)


def scatter_map(ax, pts, *, color=None, c=None, cmap="viridis", s=5, alpha=0.85, label=None, zorder=2):
    scatter_kwargs = dict(s=s, alpha=alpha, linewidths=0, label=label, zorder=zorder)
    if c is None:
        scatter_kwargs["color"] = color
    else:
        scatter_kwargs["c"] = c
        scatter_kwargs["cmap"] = cmap
    ax.scatter(pts[:, 0], pts[:, 1], **scatter_kwargs)
    ax.set_xlim(-0.04, 1.04)
    ax.set_ylim(-0.04, 1.04)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
        spine.set_color("0.78")


X, MAPS, TRUE_COEFFS, TRUE_ATOMS, META, DIGITS = load_convex_combo_dataset(DATA_DIR)
MODEL, CKPT, CONFIG = rebuild_model(CKPT_PATH, CONFIG_PATH, DEVICE)
assert CKPT.get("method") == "displacement_centered", f"Expected centered displacement checkpoint, got {CKPT.get('method')}"
assert CKPT.get("displacement_center_mode") == "train_mean", f"Expected train_mean centering, got {CKPT.get('displacement_center_mode')}"
assert CKPT.get("model_seed") == 0, f"Expected model seed 0, got {CKPT.get('model_seed')}"
assert "displacement_center" in CKPT, "Centered checkpoint is missing its displacement center."
RECONS, CODES, RECON_LOSSES = forward_all(MODEL, MAPS, BATCH_SIZE, DEVICE)
with torch.no_grad():
    LEARNED_ATOMS = MODEL.atoms_module().detach().cpu()

TRAIN_IDX, TEST_IDX = train_test_indices(
    len(MAPS), CONFIG.get("test_fraction", 0.1), CONFIG.get("seed", 42)
)

X_NP = X.numpy()
MAPS_NP = MAPS.numpy()
RECONS_NP = RECONS.numpy()
TRUE_ATOMS_NP = TRUE_ATOMS.numpy()
LEARNED_ATOMS_NP = LEARNED_ATOMS.numpy()
TRUE_COEFFS_NP = TRUE_COEFFS.numpy()
CODES_NP = CODES.numpy()
RECON_LOSSES_NP = RECON_LOSSES.numpy()
SOURCE_COLOR = X_NP[:, 0]

print(f"repo: {REPO_ROOT}")
print(f"checkpoint: {CKPT_PATH.relative_to(REPO_ROOT)} ({CKPT.get('checkpoint_role', 'unknown')})")
print(f"run config: l1={CONFIG.get('sparsity_coeffs')} lr={CONFIG.get('lr')} batch={CONFIG.get('batch_size')} lista_steps={CONFIG.get('lista_steps')} scheduler={CONFIG.get('scheduler')}")
print(f"digits: {DIGITS}; planted MNIST sample index: {META['digit_sample_index']}")
print(f"synthetic maps: {tuple(MAPS.shape)}; true atoms: {tuple(TRUE_ATOMS.shape)}; learned atoms: {tuple(LEARNED_ATOMS.shape)}")
print(f"train/test split: {len(TRAIN_IDX)} train, {len(TEST_IDX)} test random synthetic samples; no digit class is held out")
print(f"mean reconstruction loss, all:  {RECON_LOSSES_NP.mean():.6g}")
print(f"mean reconstruction loss, test: {RECON_LOSSES_NP[TEST_IDX].mean():.6g}")


## Available 02378 Run Summaries

This cell gathers saved eval summaries for the local 02378 runs, including the leakage-free `train_mean` run. Use it to sanity-check which trial the notebook is analyzing.


In [ ]:
# AVAILABLE_02378_RUN_SUMMARIES
summary_rows = []
for summary_path in sorted((REPO_ROOT / "mnist" / "results").glob("convex_combos_02378*/eval/summary.csv")):
    df = pd.read_csv(summary_path)
    if df.empty:
        continue
    best_rows = df[df["checkpoint_role"].astype(str).str.lower().eq("best")]
    if best_rows.empty:
        best_rows = df
    row = best_rows.sort_values("matched_mse").iloc[0].to_dict()
    row["result_set"] = summary_path.parents[1].name
    row["summary_csv"] = str(summary_path.relative_to(REPO_ROOT))
    summary_rows.append(row)

available_02378_summaries = pd.DataFrame(summary_rows)
cols = [
    "result_set", "checkpoint_role", "l1", "lr", "batch_size", "lista_steps",
    "scheduler", "recon_loss", "mean_active", "matched_mse",
    "mse_digit_0", "mse_digit_2", "mse_digit_3", "mse_digit_7", "mse_digit_8",
]
cols = [c for c in cols if c in available_02378_summaries.columns]
display(
    available_02378_summaries[cols]
    .sort_values("matched_mse")
    .style.format({
        "l1": "{:.6g}", "lr": "{:.6g}", "recon_loss": "{:.6g}",
        "mean_active": "{:.3f}", "matched_mse": "{:.6g}",
        "mse_digit_0": "{:.6g}", "mse_digit_2": "{:.6g}",
        "mse_digit_3": "{:.6g}", "mse_digit_7": "{:.6g}",
        "mse_digit_8": "{:.6g}",
    })
    .hide(axis="index")
)


## Planted Map Nearest Neighbors Among MNIST Maps

As a quick sanity check before looking at learned atoms, this cell uses each planted MNIST transport map as the query and reports the top five nearest raw MNIST transport maps across all digits under squared `L2(rho)`, excluding the planted map itself.


In [ ]:
# Top-5 nearest raw MNIST OT maps for each planted map, not learned atoms.
PLANTED_MNIST_TOPK = 5
planted_sample_index = int(META["digit_sample_index"])

mnist_base_for_planted_nn = torch.load(MNIST_OT_DIR / "base_measure.pt", map_location="cpu").float().numpy()
assert np.allclose(mnist_base_for_planted_nn, X_NP), "MNIST OT base measure does not match convex-combo base measure."

mnist_maps_by_digit_for_planted_nn = {
    digit: torch.load(MNIST_OT_DIR / f"digit_{digit}" / "mappings.pt", map_location="cpu").float().numpy()
    for digit in range(10)
}
all_mnist_maps_for_planted_nn = np.concatenate(
    [mnist_maps_by_digit_for_planted_nn[digit] for digit in range(10)], axis=0,
)
all_mnist_digits_for_planted_nn = np.concatenate([
    np.full(len(mnist_maps_by_digit_for_planted_nn[digit]), digit, dtype=int)
    for digit in range(10)
])
all_mnist_indices_for_planted_nn = np.concatenate([
    np.arange(len(mnist_maps_by_digit_for_planted_nn[digit]), dtype=int)
    for digit in range(10)
])

planted_mnist_neighbor_rows = []
TOP5_MNIST_BY_PLANTED_MAP = {}

for true_j, digit in enumerate(DIGITS):
    losses = l2rho_sq_np(all_mnist_maps_for_planted_nn, TRUE_ATOMS_NP[true_j])
    is_self = (
        (all_mnist_digits_for_planted_nn == digit)
        & (all_mnist_indices_for_planted_nn == planted_sample_index)
    )
    losses = losses.copy()
    losses[is_self] = np.inf
    top_idx = np.argsort(losses)[:PLANTED_MNIST_TOPK]
    TOP5_MNIST_BY_PLANTED_MAP[true_j] = top_idx

    for rank, idx in enumerate(top_idx, start=1):
        planted_mnist_neighbor_rows.append({
            "planted_map_col": true_j,
            "planted_digit": digit,
            "planted_index": planted_sample_index,
            "rank": rank,
            "neighbor_digit": int(all_mnist_digits_for_planted_nn[idx]),
            "neighbor_index": int(all_mnist_indices_for_planted_nn[idx]),
            "L2rho_sq": float(losses[idx]),
        })

planted_mnist_top5_df = pd.DataFrame(planted_mnist_neighbor_rows)
print(
    f"Top {PLANTED_MNIST_TOPK} nearest raw MNIST OT maps for each planted map "
    f"out of {len(all_mnist_maps_for_planted_nn)} maps, excluding the planted map itself."
)
display(planted_mnist_top5_df.style.format({"L2rho_sq": "{:.6g}"}))

fig_planted_nn, axes_planted_nn = plt.subplots(
    len(DIGITS),
    PLANTED_MNIST_TOPK + 1,
    figsize=(1.35 * (PLANTED_MNIST_TOPK + 1), 1.35 * len(DIGITS)),
    squeeze=False,
    constrained_layout=True,
)

for row, digit in enumerate(DIGITS):
    ax = axes_planted_nn[row, 0]
    scatter_map(ax, TRUE_ATOMS_NP[row], color=ALL_DIGIT_COLORS[digit], s=5, alpha=0.86)
    ax.set_title(f"Planted {digit}\nidx {planted_sample_index}", pad=7, fontsize=7)

    losses = l2rho_sq_np(all_mnist_maps_for_planted_nn, TRUE_ATOMS_NP[row])
    losses = losses.copy()
    losses[
        (all_mnist_digits_for_planted_nn == digit)
        & (all_mnist_indices_for_planted_nn == planted_sample_index)
    ] = np.inf
    for col, idx in enumerate(TOP5_MNIST_BY_PLANTED_MAP[row], start=1):
        ax = axes_planted_nn[row, col]
        neighbor_digit = int(all_mnist_digits_for_planted_nn[idx])
        neighbor_index = int(all_mnist_indices_for_planted_nn[idx])
        scatter_map(
            ax,
            all_mnist_maps_for_planted_nn[idx],
            color=ALL_DIGIT_COLORS[neighbor_digit],
            s=5,
            alpha=0.84,
        )
        ax.set_title(f"#{col}: digit {neighbor_digit}\nidx {neighbor_index}", pad=7, fontsize=7)
        ax.text(0.5, -0.065, f"MSE: {losses[idx]:.1e}", transform=ax.transAxes, ha="center", va="top", fontsize=7)

planted_nn_png = OUT_DIR / f"{FIG_PREFIX}_planted_map_mnist_top5.png"
planted_nn_pdf = OUT_DIR / f"{FIG_PREFIX}_planted_map_mnist_top5.pdf"
fig_planted_nn.savefig(planted_nn_png, bbox_inches="tight")
fig_planted_nn.savefig(planted_nn_pdf, bbox_inches="tight")
print(f"saved: {planted_nn_png}")
print(f"saved: {planted_nn_pdf}")
plt.show()


## Match Atoms And Measure Coefficient Recovery

The learned atoms are matched to the five original atoms using a Hungarian assignment with squared `L2(rho)` atom distance as the global cost. Coefficients are then simplex-normalized and reordered into digit order before computing MSE/RMSE/MAE against the original coefficients.


In [ ]:

def match_atoms_by_l2(learned_atoms, true_atoms):
    m, k = learned_atoms.shape[0], true_atoms.shape[0]
    cost = np.zeros((m, k), dtype=np.float64)
    for learned_j in range(m):
        for true_j in range(k):
            cost[learned_j, true_j] = float(l2rho_sq_np(learned_atoms[learned_j], true_atoms[true_j]))
    row_ind, col_ind = linear_sum_assignment(cost)
    true_to_learned = {int(true_j): int(learned_j) for learned_j, true_j in zip(row_ind, col_ind)}
    learned_to_true = {int(learned_j): int(true_j) for learned_j, true_j in zip(row_ind, col_ind)}
    return {"cost": cost, "true_to_learned": true_to_learned, "learned_to_true": learned_to_true}


MATCHING = match_atoms_by_l2(LEARNED_ATOMS_NP, TRUE_ATOMS_NP)
CODE_SIMPLEX = simplex_normalize(np.clip(CODES_NP, 0.0, None))
MATCHED_CODES = np.zeros_like(TRUE_COEFFS_NP)
for true_j, learned_j in MATCHING["true_to_learned"].items():
    MATCHED_CODES[:, true_j] = CODE_SIMPLEX[:, learned_j]

COEFF_ERROR = MATCHED_CODES - TRUE_COEFFS_NP

matching_rows = []
for true_j, digit in enumerate(DIGITS):
    learned_j = MATCHING["true_to_learned"][true_j]
    matching_rows.append({
        "digit": digit,
        "true_atom_col": true_j,
        "learned_atom": learned_j,
        "atom_L2rho_sq": MATCHING["cost"][learned_j, true_j],
    })
matching_df = pd.DataFrame(matching_rows)

summary_rows = []
for split_name, idx in [
    ("all", np.arange(len(TRUE_COEFFS_NP))),
    ("train", TRAIN_IDX),
    ("test", TEST_IDX),
]:
    err = COEFF_ERROR[idx]
    summary_rows.append({
        "split": split_name,
        "n": len(idx),
        "mse": float(np.mean(err ** 2)),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "mae": float(np.mean(np.abs(err))),
        "max_abs": float(np.max(np.abs(err))),
    })
coeff_summary_df = pd.DataFrame(summary_rows)

per_digit_df = pd.DataFrame({
    "digit": DIGITS,
    "mse": np.mean(COEFF_ERROR ** 2, axis=0),
    "rmse": np.sqrt(np.mean(COEFF_ERROR ** 2, axis=0)),
    "mae": np.mean(np.abs(COEFF_ERROR), axis=0),
})

atom_cost_df = pd.DataFrame(
    MATCHING["cost"],
    index=[f"learned {j}" for j in range(LEARNED_ATOMS_NP.shape[0])],
    columns=[f"digit {d}" for d in DIGITS],
)

print("Hungarian atom matching: learned atom -> planted digit")
display(matching_df)
print("Atom matching cost matrix: squared L2(rho) between transport maps")
display(atom_cost_df.style.format("{:.6g}"))
print("Average coefficient recovery loss")
display(coeff_summary_df.style.format({"mse": "{:.6g}", "rmse": "{:.6g}", "mae": "{:.6g}", "max_abs": "{:.6g}"}))
print("Per-digit coefficient recovery loss, all samples")
display(per_digit_df.style.format({"mse": "{:.6g}", "rmse": "{:.6g}", "mae": "{:.6g}"}))


## Coefficient MSE Quick Report

This cell reports the coefficient recovery MSE directly. Codes are clipped to nonnegative values, normalized onto the simplex, matched to planted atoms by the Hungarian assignment above, then compared against the true convex-combo coefficients.


In [ ]:
# COEFFICIENT_MSE_QUICK_REPORT
split_mse = coeff_summary_df[["split", "n", "mse", "rmse", "mae"]].copy()
split_mse["section"] = "overall"
split_mse = split_mse[["section", "split", "n", "mse", "rmse", "mae"]]

digit_mse = per_digit_df[["digit", "mse", "rmse", "mae"]].copy()
digit_mse.insert(0, "section", "per digit")
digit_mse = digit_mse.rename(columns={"digit": "split"})
digit_mse["split"] = digit_mse["split"].map(lambda d: f"digit {int(d)}")
digit_mse["n"] = len(TRUE_COEFFS_NP)
digit_mse = digit_mse[["section", "split", "n", "mse", "rmse", "mae"]]

quick_coeff_mse = pd.concat([split_mse, digit_mse], ignore_index=True)

test_mse = float(coeff_summary_df.loc[coeff_summary_df["split"] == "test", "mse"].iloc[0])
all_mse = float(coeff_summary_df.loc[coeff_summary_df["split"] == "all", "mse"].iloc[0])
print(f"Coefficient MSE: test={test_mse:.6g}  all={all_mse:.6g}")
display(
    quick_coeff_mse.style
    .format({"mse": "{:.6g}", "rmse": "{:.6g}", "mae": "{:.6g}"})
    .hide(axis="index")
)


## Coefficient Recovery Diagnostics

For the five-digit run, the coefficient space is too high-dimensional for the old simplex view. These heatmaps keep the same recovery question visible: do matched learned codes put mass on the same two planted digits, and where do errors concentrate?


In [ ]:
# Visual coefficient diagnostics replacing the old low-dimensional simplex view.
active_digit_counts = (TRUE_COEFFS_NP > 1e-8).sum(axis=0)
pair_counts = np.zeros((len(DIGITS), len(DIGITS)), dtype=int)
for row in TRUE_COEFFS_NP:
    active = np.flatnonzero(row > 1e-8)
    if len(active) == 2:
        i, j = active
        pair_counts[i, j] += 1
        pair_counts[j, i] += 1

# Sort held-out examples by coefficient recovery error for a compact visual audit.
test_coeff_mse = np.mean(COEFF_ERROR[TEST_IDX] ** 2, axis=1)
best_test = TEST_IDX[np.argsort(test_coeff_mse)[:25]]
worst_test = TEST_IDX[np.argsort(test_coeff_mse)[-25:][::-1]]

fig_coeff, axes_coeff = plt.subplots(2, 3, figsize=(11.0, 6.5), constrained_layout=True)

ax = axes_coeff[0, 0]
im = ax.imshow(MATCHING["cost"], cmap="magma_r")
ax.set_title("Atom L2(rho) matching cost")
ax.set_xlabel("Planted digit")
ax.set_ylabel("Learned atom")
ax.set_xticks(np.arange(len(DIGITS)))
ax.set_xticklabels(DIGITS)
ax.set_yticks(np.arange(LEARNED_ATOMS_NP.shape[0]))
fig_coeff.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes_coeff[0, 1]
vals = per_digit_df["mse"].to_numpy()
ax.bar(np.arange(len(DIGITS)), vals, color=[ALL_DIGIT_COLORS[d] for d in DIGITS])
ax.set_title("Coefficient MSE by digit")
ax.set_xlabel("Digit")
ax.set_ylabel("MSE")
ax.set_xticks(np.arange(len(DIGITS)))
ax.set_xticklabels(DIGITS)

ax = axes_coeff[0, 2]
im = ax.imshow(pair_counts, cmap="Blues")
ax.set_title("2-sparse pair counts")
ax.set_xlabel("Digit")
ax.set_ylabel("Digit")
ax.set_xticks(np.arange(len(DIGITS)))
ax.set_xticklabels(DIGITS)
ax.set_yticks(np.arange(len(DIGITS)))
ax.set_yticklabels(DIGITS)
fig_coeff.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for ax, idx, title in [
    (axes_coeff[1, 0], best_test, "Best 25 held-out: true coeffs"),
    (axes_coeff[1, 1], best_test, "Best 25 held-out: matched codes"),
    (axes_coeff[1, 2], worst_test, "Worst 25 held-out: coeff error"),
]:
    if "true" in title:
        mat = TRUE_COEFFS_NP[idx]
        cmap = "viridis"
        vlim = (0.0, 1.0)
    elif "matched" in title:
        mat = MATCHED_CODES[idx]
        cmap = "viridis"
        vlim = (0.0, 1.0)
    else:
        mat = COEFF_ERROR[idx]
        cmap = "coolwarm"
        vmax = np.max(np.abs(mat))
        vlim = (-vmax, vmax)
    im = ax.imshow(mat, aspect="auto", cmap=cmap, vmin=vlim[0], vmax=vlim[1])
    ax.set_title(title)
    ax.set_xlabel("Digit")
    ax.set_ylabel("Example")
    ax.set_xticks(np.arange(len(DIGITS)))
    ax.set_xticklabels(DIGITS)
    ax.set_yticks([])
    fig_coeff.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

coeff_png = OUT_DIR / f"{FIG_PREFIX}_coeff_diagnostics.png"
coeff_pdf = OUT_DIR / f"{FIG_PREFIX}_coeff_diagnostics.pdf"
fig_coeff.savefig(coeff_png, bbox_inches="tight")
fig_coeff.savefig(coeff_pdf, bbox_inches="tight")
print(f"saved: {coeff_png}")
print(f"saved: {coeff_pdf}")
plt.show()


## Figure 1a/1b: Atoms And Reconstructions

The atom figure shows original atoms over matched learned atoms, as in the three-digit notebook, now expanded to all five planted digits. The reconstruction figure shows a deterministic set of balanced held-out synthetic maps spanning all digit pairs among 0,2,4,6,8. Titles are kept minimal; distances/errors are printed below the relevant panels.


In [ ]:
def choose_reconstruction_examples(candidate_idx, recon_losses, coeffs, required_pairs=None, balance_range=(0.3, 0.7), active_tol=1e-8):
    """Pick deterministic held-out examples for balanced active coefficient pairs."""
    if required_pairs is None:
        required_pairs = [(i, j) for i in range(len(DIGITS)) for j in range(i + 1, len(DIGITS))]
    candidate_idx = np.array(candidate_idx, dtype=int)
    selected = []
    example_rows = []
    used = set()
    for pair in required_pairs:
        pair = tuple(pair)
        pair_set = set(pair)
        exact_balanced = []
        exact_any = []
        for idx in candidate_idx:
            if int(idx) in used:
                continue
            active = set(np.flatnonzero(np.abs(coeffs[idx]) > active_tol).tolist())
            if active != pair_set:
                continue
            exact_any.append(int(idx))
            pair_values = coeffs[idx, list(pair)]
            balanced = np.all((pair_values >= balance_range[0]) & (pair_values <= balance_range[1]))
            if balanced:
                exact_balanced.append(int(idx))
        pool = exact_balanced if exact_balanced else exact_any
        if not pool:
            pair_digits = tuple(DIGITS[j] for j in pair)
            raise ValueError(f"No held-out example found for coefficient pair {pair_digits}.")
        ordered = np.array(pool, dtype=int)[np.argsort(recon_losses[pool])]
        chosen = int(ordered[len(ordered) // 2])
        selected.append(chosen)
        used.add(chosen)
        example_rows.append({
            "pair": tuple(DIGITS[j] for j in pair),
            "index": chosen,
            "n_pair_test": len(exact_any),
            "n_balanced_test": len(exact_balanced),
            "coeff_a": float(coeffs[chosen, pair[0]]),
            "coeff_b": float(coeffs[chosen, pair[1]]),
            "recon_loss": float(recon_losses[chosen]),
            "used_balanced_pool": bool(exact_balanced),
        })
    return np.array(selected, dtype=int), pd.DataFrame(example_rows)


def panel_note(ax, text, note_y=-0.10):
    ax.text(0.5, note_y, text, transform=ax.transAxes, ha="center", va="top", fontsize=7)


all_example_pairs = [(i, j) for i in range(len(DIGITS)) for j in range(i + 1, len(DIGITS))]
EXAMPLE_PAIRS = [all_example_pairs[i] for i in np.linspace(0, len(all_example_pairs) - 1, 3, dtype=int)]
EXAMPLE_IDX, example_selection_df = choose_reconstruction_examples(
    TEST_IDX, RECON_LOSSES_NP, TRUE_COEFFS_NP, required_pairs=EXAMPLE_PAIRS,
)
print("selected reconstruction examples:")
display(example_selection_df.style.format({
    "coeff_a": "{:.3f}",
    "coeff_b": "{:.3f}",
    "recon_loss": "{:.2e}",
}))
original_color = TARGET_COLOR
learned_color = RECON_COLOR
overlay_original_color = "#356AA0"
overlay_reconstruction_color = "#B33F62"
atom_alpha = 0.46
reconstruction_alpha = 0.82
source_alpha = 0.56
map_line_alpha = 0.2


def overlay_measure_reconstruction_map(ax, source_pts, original_pts, recon_pts, *, stride=4):
    sample = np.arange(0, len(source_pts), stride)
    for i in sample:
        arrow = mpl.patches.FancyArrowPatch(
            (source_pts[i, 0], source_pts[i, 1]),
            (recon_pts[i, 0], recon_pts[i, 1]),
            arrowstyle="-|>",
            mutation_scale=5.0,
            color=overlay_reconstruction_color,
            alpha=map_line_alpha,
            linewidth=0.65,
            shrinkA=0.0,
            shrinkB=0.0,
            zorder=1,
        )
        ax.add_patch(arrow)
    scatter_map(ax, source_pts, color="#707070", s=4.2, alpha=source_alpha, label="Source", zorder=2)
    scatter_map(ax, original_pts, color=overlay_original_color, s=5.6, alpha=0.58, label="Original", zorder=3)
    scatter_map(ax, recon_pts, color=overlay_reconstruction_color, s=5.6, alpha=0.58, label="Reconstruction", zorder=4)

# Figure 1a: atoms only.
n_atoms = len(DIGITS)
fig_atoms, axes_atoms = plt.subplots(2, n_atoms, figsize=(1.38 * n_atoms, 3.1), constrained_layout=True, squeeze=False)
for true_j in range(n_atoms):
    digit = DIGITS[true_j]
    learned_j = MATCHING["true_to_learned"][true_j]

    ax = axes_atoms[0, true_j]
    scatter_map(ax, TRUE_ATOMS_NP[true_j], color=ALL_DIGIT_COLORS[digit], s=5.5, alpha=atom_alpha)
    ax.set_title(f"True {digit}", pad=7)

    ax = axes_atoms[1, true_j]
    scatter_map(ax, LEARNED_ATOMS_NP[learned_j], color=learned_color, s=5.5, alpha=atom_alpha)
    ax.set_title(f"Learned {learned_j}", pad=7)
    panel_note(ax, f"MSE: {MATCHING['cost'][learned_j, true_j]:.1e}")

atoms_png = OUT_DIR / f"{FIG_PREFIX}_atoms.png"
atoms_pdf = OUT_DIR / f"{FIG_PREFIX}_atoms.pdf"
fig_atoms.savefig(atoms_png, bbox_inches="tight")
fig_atoms.savefig(atoms_pdf, bbox_inches="tight")
print(f"saved: {atoms_png}")
print(f"saved: {atoms_pdf}")
plt.show()

# Figure 1b: data/reconstructions only. Three two-sparse digit pairs are shown.
n_examples = len(EXAMPLE_IDX)
n_cols = n_examples
n_bands = int(np.ceil(n_examples / n_cols))
fig_recon, axes_recon = plt.subplots(2 * n_bands, n_cols, figsize=(1.38 * n_cols, 2.75 * n_bands), constrained_layout=True, squeeze=False)
for ax in axes_recon.ravel():
    ax.axis("off")

for panel, idx in enumerate(EXAMPLE_IDX):
    band = panel // n_cols
    col = panel % n_cols
    pair = tuple(np.flatnonzero(TRUE_COEFFS_NP[idx] > 1e-8).tolist())
    pair_digits = tuple(DIGITS[j] for j in pair)

    ax = axes_recon[2 * band, col]
    ax.axis("on")
    scatter_map(ax, MAPS_NP[idx], color=original_color, s=5.5, alpha=reconstruction_alpha)
    ax.set_title(f"{pair_digits[0]}+{pair_digits[1]}", pad=7)

    ax = axes_recon[2 * band + 1, col]
    ax.axis("on")
    scatter_map(ax, RECONS_NP[idx], color=learned_color, s=5.5, alpha=reconstruction_alpha)
    ax.set_title(f"Recon {panel + 1}", pad=7)
    panel_note(ax, f"MSE: {RECON_LOSSES_NP[idx]:.1e}")

recon_png = OUT_DIR / f"{FIG_PREFIX}_reconstructions.png"
recon_pdf = OUT_DIR / f"{FIG_PREFIX}_reconstructions.pdf"
fig_recon.savefig(recon_png, bbox_inches="tight")
fig_recon.savefig(recon_pdf, bbox_inches="tight")
print(f"saved: {recon_png}")
print(f"saved: {recon_pdf}")
print("example coefficients:")
display(pd.DataFrame(TRUE_COEFFS_NP[EXAMPLE_IDX], index=EXAMPLE_IDX, columns=[f"digit_{d}" for d in DIGITS]))
plt.show()


## Figure 1d: Matched Atom Vector Fields

These panels show the source-to-target displacement field `T(x) - x` for each planted atom and its matched learned atom. The arrows are subsampled for readability but preserve the shared base-measure correspondence.


In [ ]:
def plot_vector_field(ax, source_pts, target_pts, *, color, title, stride=4, scale=1.0):
    sample = np.arange(0, len(source_pts), stride)
    delta = target_pts - source_pts
    ax.quiver(
        source_pts[sample, 0],
        source_pts[sample, 1],
        delta[sample, 0],
        delta[sample, 1],
        angles="xy",
        scale_units="xy",
        scale=scale,
        color=color,
        width=0.0032,
        alpha=0.76,
    )
    scatter_map(ax, source_pts, color=BASE_COLOR, s=3.5, alpha=0.38, zorder=1)
    ax.set_title(title, pad=7)


field_stride = 4
fig_fields, axes_fields = plt.subplots(2, len(DIGITS), figsize=(1.52 * len(DIGITS), 3.25), constrained_layout=True, squeeze=False)
for true_j, digit in enumerate(DIGITS):
    learned_j = MATCHING["true_to_learned"][true_j]
    plot_vector_field(
        axes_fields[0, true_j],
        X_NP,
        TRUE_ATOMS_NP[true_j],
        color=ALL_DIGIT_COLORS[digit],
        title=f"True {digit}",
        stride=field_stride,
    )
    plot_vector_field(
        axes_fields[1, true_j],
        X_NP,
        LEARNED_ATOMS_NP[learned_j],
        color=learned_color,
        title=f"Learned {learned_j}",
        stride=field_stride,
    )
    panel_note(axes_fields[1, true_j], f"MSE: {MATCHING['cost'][learned_j, true_j]:.1e}")

fields_png = OUT_DIR / f"{FIG_PREFIX}_atom_vector_fields.png"
fields_pdf = OUT_DIR / f"{FIG_PREFIX}_atom_vector_fields.pdf"
fig_fields.savefig(fields_png, bbox_inches="tight")
fig_fields.savefig(fields_pdf, bbox_inches="tight")
print(f"saved: {fields_png}")
print(f"saved: {fields_pdf}")
plt.show()


In [ ]:

# Figure 1c: one source measure, original target, reconstruction, and source-to-reconstruction map overlay.
overlay_idx = int(EXAMPLE_IDX[len(EXAMPLE_IDX) // 2])
fig_overlay, ax_overlay = plt.subplots(1, 1, figsize=(3.85, 3.25), constrained_layout=True)
overlay_measure_reconstruction_map(ax_overlay, X_NP, MAPS_NP[overlay_idx], RECONS_NP[overlay_idx], stride=1)
handles, labels = ax_overlay.get_legend_handles_labels()
fig_overlay.legend(handles, labels, loc="center left", ncol=1, frameon=False, bbox_to_anchor=(0.88, 0.5))
overlay_png = OUT_DIR / f"{FIG_PREFIX}_overlay_map.png"
overlay_pdf = OUT_DIR / f"{FIG_PREFIX}_overlay_map.pdf"
fig_overlay.savefig(overlay_png, bbox_inches="tight")
fig_overlay.savefig(overlay_pdf, bbox_inches="tight")
print(f"saved: {overlay_png}")
print(f"saved: {overlay_pdf}")
plt.show()

## Figure 2: Nearest Held-Out MNIST Transport Maps

For each matched learned atom, this cell searches a deterministic held-out MNIST pool: 10% of maps from every digit 0-9, using the same `test_fraction` and `seed` convention as the SAE run. The figure shows only losses; the table reports neighbor digit/index details.


In [ ]:
def load_digit_maps(mnist_ot_dir, digit):
    return torch.load(Path(mnist_ot_dir) / f"digit_{digit}" / "mappings.pt", map_location="cpu").float().numpy()


def heldout_indices_for_digit(n, test_fraction, seed, digit):
    # Offset by digit so each digit gets a deterministic but distinct split.
    _, test_idx = train_test_indices(n, test_fraction, int(seed) + int(digit))
    return np.array(test_idx, dtype=int)


mnist_base = torch.load(MNIST_OT_DIR / "base_measure.pt", map_location="cpu").float().numpy()
assert np.allclose(mnist_base, X_NP), "MNIST OT base measure does not match convex-combo base measure."

mnist_test_fraction = CONFIG.get("test_fraction", 0.1)
mnist_seed = CONFIG.get("seed", 42)
original_idx = int(META["digit_sample_index"])
all_digit_maps = {digit: load_digit_maps(MNIST_OT_DIR, digit) for digit in range(10)}


def build_mnist_search_pool(search_digits, *, include_original_refs=True):
    search_digits = [int(d) for d in search_digits]
    heldout_maps = []
    heldout_digits = []
    heldout_sample_indices = []
    heldout_is_planted_ref = []

    for digit in search_digits:
        maps_d = all_digit_maps[digit]
        idx_d = heldout_indices_for_digit(len(maps_d), mnist_test_fraction, mnist_seed, digit)
        idx_d = idx_d[idx_d != original_idx]
        heldout_maps.append(maps_d[idx_d])
        heldout_digits.append(np.full(len(idx_d), digit, dtype=int))
        heldout_sample_indices.append(idx_d)
        heldout_is_planted_ref.append(np.zeros(len(idx_d), dtype=bool))

    if include_original_refs:
        planted_ref_digits = [digit for digit in DIGITS if digit in search_digits]
        if planted_ref_digits:
            planted_ref_maps = np.stack([all_digit_maps[d][original_idx] for d in planted_ref_digits], axis=0)
            heldout_maps.append(planted_ref_maps)
            heldout_digits.append(np.array(planted_ref_digits, dtype=int))
            heldout_sample_indices.append(np.full(len(planted_ref_digits), original_idx, dtype=int))
            heldout_is_planted_ref.append(np.ones(len(planted_ref_digits), dtype=bool))

    return {
        "maps": np.concatenate(heldout_maps, axis=0),
        "digits": np.concatenate(heldout_digits, axis=0),
        "indices": np.concatenate(heldout_sample_indices, axis=0),
        "is_planted_ref": np.concatenate(heldout_is_planted_ref, axis=0),
        "search_digits": tuple(search_digits),
    }


def run_nearest_neighbor_search(pool, *, label, filename_suffix):
    search_rows = []
    summary_rows = []
    top5_by_atom = {}

    for true_j, digit in enumerate(DIGITS):
        learned_j = MATCHING["true_to_learned"][true_j]
        learned_atom = LEARNED_ATOMS_NP[learned_j]
        losses = l2rho_sq_np(pool["maps"], learned_atom)
        order = np.argsort(losses)
        top5 = order[:5]
        top5_by_atom[true_j] = top5

        top_digits = pool["digits"][top5].tolist()
        summary_rows.append({
            "atom": true_j + 1,
            "matched_learned_atom": learned_j,
            "matched_original_digit": digit,
            "top5_digits": top_digits,
            "all_top5_same_as_matched_digit": all(d == digit for d in top_digits),
        })
        for rank, pool_idx in enumerate(top5, start=1):
            search_rows.append({
                "atom": true_j + 1,
                "matched_learned_atom": learned_j,
                "rank": rank,
                "neighbor_digit": int(pool["digits"][pool_idx]),
                "neighbor_index": int(pool["indices"][pool_idx]),
                "L2rho_sq": float(losses[pool_idx]),
                "is_planted_reference": bool(pool["is_planted_ref"][pool_idx]),
            })

    summary_df = pd.DataFrame(summary_rows)
    top5_df = pd.DataFrame(search_rows)

    n_planted_refs = int(pool["is_planted_ref"].sum())
    print(f"{label}: {pool['maps'].shape[0]} maps ({int(100 * mnist_test_fraction)}% per digit, excluding sample index {original_idx}, + {n_planted_refs} planted refs).")
    display(summary_df)
    display(top5_df.style.format({"L2rho_sq": "{:.6g}"}))

    fig, axes = plt.subplots(len(DIGITS), 6, figsize=(8.0, 1.45 * len(DIGITS)), squeeze=False, constrained_layout=True)
    for row, digit in enumerate(DIGITS):
        true_j = row
        learned_j = MATCHING["true_to_learned"][true_j]

        ax0 = axes[row, 0]
        scatter_map(ax0, LEARNED_ATOMS_NP[learned_j], color=learned_color, s=5, alpha=0.86)
        ax0.set_title(f"Learned {learned_j}\nmatched digit {digit}", pad=7, fontsize=7)

        learned_atom = LEARNED_ATOMS_NP[learned_j]
        losses = l2rho_sq_np(pool["maps"], learned_atom)
        for col, pool_idx in enumerate(top5_by_atom[true_j], start=1):
            ax = axes[row, col]
            neighbor_digit = int(pool["digits"][pool_idx])
            scatter_map(
                ax,
                pool["maps"][pool_idx],
                color=ALL_DIGIT_COLORS.get(neighbor_digit, "#444444"),
                s=5,
                alpha=0.84,
            )
            if pool["is_planted_ref"][pool_idx]:
                for spine in ax.spines.values():
                    spine.set_color("#111111")
                    spine.set_linewidth(1.6)
            ax.set_title(f"#{col}: digit {neighbor_digit}", pad=7, fontsize=7)
            panel_note(ax, f"MSE: {losses[pool_idx]:.1e}", note_y=-0.065)

    fig.canvas.draw()
    for upper_row in range(len(DIGITS) - 1):
        upper_bottom = min(ax.get_position().y0 for ax in axes[upper_row, :])
        lower_top = max(ax.get_position().y1 for ax in axes[upper_row + 1, :])
        y_sep = 0.5 * (upper_bottom + lower_top)
        fig.add_artist(mpl.lines.Line2D(
            [0.015, 0.985], [y_sep, y_sep],
            transform=fig.transFigure,
            color="0.86",
            linewidth=0.8,
            zorder=0,
        ))

    neighbor_left = axes[-1, 1].get_position().x0
    neighbor_right = axes[-1, -1].get_position().x1
    neighbor_bottom = min(ax.get_position().y0 for ax in axes[-1, 1:])
    arrow_y = neighbor_bottom - 0.085
    fig.add_artist(mpl.patches.FancyArrowPatch(
        (neighbor_left, arrow_y),
        (neighbor_right, arrow_y),
        transform=fig.transFigure,
        arrowstyle="->",
        mutation_scale=11,
        linewidth=1.0,
        color="0.20",
        clip_on=False,
    ))
    fig.text(
        0.5 * (neighbor_left + neighbor_right),
        arrow_y - 0.024,
        "Increasing MSE",
        ha="center",
        va="top",
        fontsize=8,
        color="0.20",
    )

    png = OUT_DIR / f"{FIG_PREFIX}_{filename_suffix}.png"
    pdf = OUT_DIR / f"{FIG_PREFIX}_{filename_suffix}.pdf"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"saved: {png}")
    print(f"saved: {pdf}")
    plt.show()

    return summary_df, top5_df, top5_by_atom


RESTRICTED_MNIST_POOL = build_mnist_search_pool(DIGITS)
ALL_MNIST_POOL = build_mnist_search_pool(range(10))

nearest_restricted_summary_df, nearest_restricted_top5_df, TOP5_RESTRICTED_BY_ATOM = run_nearest_neighbor_search(
    RESTRICTED_MNIST_POOL,
    label=f"Held-out MNIST nearest-neighbor pool restricted to digits {tuple(DIGITS)}",
    filename_suffix="nearest_mnist_restricted_digits_heldout",
)
nearest_all_summary_df, nearest_all_top5_df, TOP5_ALL_BY_ATOM = run_nearest_neighbor_search(
    ALL_MNIST_POOL,
    label="Held-out MNIST nearest-neighbor pool over all 10 digits",
    filename_suffix="nearest_mnist_all_digits_heldout",
)
